In [1]:
import mne
from mne.preprocessing import ICA
from typing import Optional, List, Union

class EEGPreprocessor:
    """EEG数据预处理接口类，提供标准化的脑电数据预处理流程"""
    
    def __init__(self, 
                 exclude_channels: List[str] = ['BIP1'],
                 montage_kind: str = 'standard_1005',
                 target_sfreq: int = 500,
                 l_freq: float = 0.5,
                 h_freq: float = 90,
                 notch_freq: Union[float, List[float], None] = 50,
                 n_ica_components: int = 15,
                 ica_max_iter: int = 2000,
                 ica_random_state: int = 97,
                 start_delay: float = 30,
                 duration: float = 300,
                 reference_method: Union[str, List[str]] = 'average',
                 eog_channels: List[str] = ['Fp1', 'Fp2']):
        """
        初始化EEG预处理参数
        
        参数:
            exclude_channels: 要排除的通道名称列表
            montage_kind: 电极位置模板名称
            target_sfreq: 目标采样率(Hz)
            l_freq: 带通滤波下限(Hz)
            h_freq: 带通滤波上限(Hz)
            notch_freq: 工频陷波频率(Hz)
            n_ica_components: ICA成分数量
            ica_max_iter: ICA最大迭代次数
            ica_random_state: ICA随机种子
            start_delay: 开始时间（秒）
            duration: 持续时间（秒）
            reference_method: 参考方式，可选'average'、'mastoids'或具体通道名列表
            eog_channels: 用于检测眼动伪迹的通道列表
        """
        self.exclude_channels = exclude_channels
        self.montage_kind = montage_kind
        self.target_sfreq = target_sfreq
        self.l_freq = l_freq
        self.h_freq = h_freq
        self.notch_freq = notch_freq
        self.n_ica_components = n_ica_components
        self.ica_max_iter = ica_max_iter
        self.ica_random_state = ica_random_state
        self.start_delay = start_delay
        self.duration = duration
        self.reference_method = reference_method
        self.eog_channels = eog_channels
        
        # 预处理结果
        self.raw_clean = None
        self.ica = None
        
    def preprocess(self, raw: mne.io.Raw) -> tuple[mne.io.Raw, ICA]:
        """
        执行完整的EEG预处理流程
        
        参数:
            raw: 原始EEG数据
            
        返回:
            预处理后的EEG数据和ICA对象
        """
        # 1. 通道筛选
        raw_processed = self._filter_channels(raw.copy())
        
        # 2. 设置电极位置
        raw_processed = self._set_montage(raw_processed)
        
        # 3. 时间切片
        #raw_processed = self._crop_time(raw_processed)
        
        # 4. 降采样
        raw_processed = self._resample(raw_processed)
        
        # 5. 陷波滤波
        raw_processed = self._notch_filter(raw_processed)
        
        # 6. 带通滤波
        raw_processed = self._bandpass_filter(raw_processed)
        
        # 7. ICA去除伪迹（包括眼动）
        raw_processed, ica = self._ica_artifact_removal(raw_processed)
        
        # 8. 重参考
        raw_processed = self._re_reference(raw_processed)
        
        # 保存结果
        self.raw_clean = raw_processed
        self.ica = ica
        
        return raw_processed, ica
    
    def _filter_channels(self, raw: mne.io.Raw) -> mne.io.Raw:
        """筛选通道，排除指定通道"""
        if self.exclude_channels:
            channels_to_keep = [ch for ch in raw.ch_names if ch not in self.exclude_channels]
            raw = raw.pick_channels(channels_to_keep)
            print(f"已排除通道: {', '.join(self.exclude_channels)}")
        return raw
    
    def _set_montage(self, raw: mne.io.Raw) -> mne.io.Raw:
        """设置电极位置"""
        if self.montage_kind:
            try:
                montage = mne.channels.make_standard_montage(self.montage_kind)
                raw = raw.set_montage(montage)
                print(f"已应用电极位置模板: {self.montage_kind}")
            except ValueError:
                print(f"警告: 无法应用电极位置模板 {self.montage_kind}，将使用默认位置")
        return raw
    
    def _crop_time(self, raw: mne.io.Raw) -> mne.io.Raw:
        """时间切片，提取指定时间段"""
        end_delay = self.start_delay + self.duration
        if end_delay > raw.tmax:
            end_delay = raw.tmax
            print(f"警告: 数据长度不足，调整结束时间为 {end_delay:.2f}s")
        return raw.crop(tmin=self.start_delay, tmax=end_delay)
    
    def _resample(self, raw: mne.io.Raw) -> mne.io.Raw:
        """降采样到目标采样率"""
        if raw.info['sfreq'] != self.target_sfreq:
            raw = raw.resample(self.target_sfreq)
            print(f"已降采样至 {self.target_sfreq}Hz")
        return raw
    
    def _notch_filter(self, raw: mne.io.Raw) -> mne.io.Raw:
        """陷波滤波去除工频干扰"""
        if self.notch_freq is not None:
            raw = raw.notch_filter(
                freqs=self.notch_freq, 
                method='fir', 
                fir_design='firwin'
            )
            print(f"已去除工频干扰: {self.notch_freq}Hz")
        return raw
    
    def _bandpass_filter(self, raw: mne.io.Raw) -> mne.io.Raw:
        """带通滤波"""
        raw = raw.filter(
            l_freq=self.l_freq, 
            h_freq=self.h_freq, 
            fir_design='firwin',
            skip_by_annotation='edge'
        )
        print(f"已应用带通滤波: {self.l_freq}-{self.h_freq}Hz")
        return raw
    
    def _ica_artifact_removal(self, raw: mne.io.Raw) -> tuple[mne.io.Raw, ICA]:
        """使用ICA去除伪迹，包括眼动伪迹"""
        ica = ICA(
            n_components=self.n_ica_components, 
            max_iter=self.ica_max_iter, 
            random_state=self.ica_random_state,
            method='fastica'
        )
        ica.fit(raw)
        print(f"ICA分解完成，提取了 {self.n_ica_components} 个成分")
        
        # 检测并排除眼动伪迹成分
        eog_inds = []
        available_eog_channels = [ch for ch in self.eog_channels if ch in raw.ch_names]
        
        if available_eog_channels:
            for ch in available_eog_channels:
                inds, _ = ica.find_bads_eog(raw, ch_name=ch)
                eog_inds.extend(inds)
            
            # 去重并保留唯一的成分索引
            eog_inds = list(set(eog_inds))
            ica.exclude = eog_inds
            print(f"ICA自动检测并排除 {len(eog_inds)} 个眼电伪迹成分 "
                  f"(使用通道: {', '.join(available_eog_channels)})")
        else:
            print(f"警告: 未找到可用的眼动检测通道 {self.eog_channels}，无法自动检测眼电伪迹")
        
        return ica.apply(raw.copy()), ica
    
    def _re_reference(self, raw: mne.io.Raw) -> mne.io.Raw:
        """重参考处理"""
        if self.reference_method == 'mastoids' and all(ch in raw.ch_names for ch in ['M1', 'M2']):
            # 双乳突参考
            raw = raw.set_eeg_reference(ref_channels=['M1', 'M2'])
            print("已应用双乳突参考")
        elif self.reference_method == 'average':
            # 平均参考
            raw = raw.set_eeg_reference(ref_channels='average')
            print("已应用平均参考")
        elif isinstance(self.reference_method, list) and all(ch in raw.ch_names for ch in self.reference_method):
            # 自定义参考通道
            raw = raw.set_eeg_reference(ref_channels=self.reference_method)
            print(f"已应用自定义参考: {', '.join(self.reference_method)}")
        else:
            print(f"警告: 参考方式 {self.reference_method} 不可用，将保留当前参考")
        
        return raw
